PD- Probability of Default
LGD- Loss Given Default
EAD- Exposure at Default

Using these paramaters, Expected Loss, EL can be derived at using the following formula

$$\text{Expected Loss (EL)} = \text{PD} \times \text{LGD} \times \text{EAD}$$


In [16]:
#Import Numpy, pandas and other packages
import pandas as pd
import numpy as np
import glob
import os

#Importing Different ML techniques and packages
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import roc_auc_score, brier_score_loss, mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

#Other Imports
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

Uploading files and creating dataframes

In [17]:
files = glob.glob('*.csv')
print("Files:", files)

#Reading the files and displaying the first two rows of each file to describe the shape and available columns

for f in files:
    try:
        df = pd.read_csv(f)
        print(f"\n--- {f} ---")
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print(df.head(2))
    except Exception as e:
        print(f"Error reading {f}: {e}")

Files: ['loan_portfolio.csv', 'vintage_analysis.csv', 'macro_stress_scenarios.csv', 'payment_history.csv', 'payment_history_dictionary.csv', 'credit_ratings.csv', 'portfolio_metrics.csv']

--- loan_portfolio.csv ---
Shape: (50000, 24)
Columns: ['loan_id', 'origination_date', 'maturity_date', 'maturity_months', 'sector', 'loan_type', 'collateral', 'initial_rating', 'credit_score', 'ead', 'coupon_rate', 'leverage', 'interest_coverage', 'debt_to_equity', 'pd_annual', 'lgd', 'el', 'unexpected_loss', 'rwa', 'defaulted', 'default_date', 'survival_months', 'recovery_rate', 'loss_given_default']
   loan_id origination_date maturity_date  maturity_months      sector  \
0  L000001       2017-10-01    2019-10-01               24  Technology   
1  L000002       2016-06-01    2020-06-01               48  Healthcare   

   loan_type collateral initial_rating  credit_score         ead  ...  \
0   mortgage    secured              B           704  1035611.99  ...   
1  term_loan  unsecured             

In [18]:
# Load main loan dataset
df_loans = pd.read_csv('loan_portfolio.csv')
print("Loan Portfolio Shape:", df_loans.shape)
print("Default rate:", df_loans['defaulted'].mean())

# Missing values check
print("Missing values in loan_portfolio:\n", df_loans.isnull().sum())

# Examine target variables:
# 1. PD target: 'defaulted'
# 2. LGD target: for defaulted loans, 'loss_given_default' or 1 - 'recovery_rate'
df_defaulted = df_loans[df_loans['defaulted'] == 1]
print("Defaulted loans count:", len(df_defaulted))
print("LGD summary on defaulted loans:\n", df_defaulted[['loss_given_default', 'recovery_rate', 'lgd']].describe())

# Check EAD: is ead actual exposure at default / origination?
print("EAD summary:\n", df_loans['ead'].describe())

Loan Portfolio Shape: (50000, 24)
Default rate: 0.139
Missing values in loan_portfolio:
 loan_id                   0
origination_date          0
maturity_date             0
maturity_months           0
sector                    0
loan_type                 0
collateral                0
initial_rating            0
credit_score              0
ead                       0
coupon_rate               0
leverage                  0
interest_coverage         0
debt_to_equity            0
pd_annual                 0
lgd                       0
el                        0
unexpected_loss           0
rwa                       0
defaulted                 0
default_date          43050
survival_months           0
recovery_rate         43050
loss_given_default    43050
dtype: int64
Defaulted loans count: 6950
LGD summary on defaulted loans:
        loss_given_default  recovery_rate          lgd
count        6.950000e+03    6950.000000  6950.000000
mean         1.713657e+06       0.454562     0.545438
std

In [19]:
# Let's inspect payment history to see if CCF or EAD model can be built, or if EAD = outstanding_balance
df_pay = pd.read_csv('payment_history.csv')
print("Payment history columns:", df_pay.columns.tolist())
print(df_pay.head())

# Check macro_stress_scenarios
df_macro = pd.read_csv('macro_stress_scenarios.csv')
print("\nMacro scenarios structure:")
print(df_macro.head(10))

Payment history columns: ['loan_id', 'month_on_book', 'payment_date', 'scheduled_payment', 'actual_payment', 'outstanding_balance', 'interest_component', 'principal_component', 'dpd', 'dpd_bucket', 'missed_payment', 'cumulative_missed', 'utilization']
   loan_id  month_on_book payment_date  scheduled_payment  actual_payment  \
0  L000003              1   2023-08-01           51808.14        52711.92   
1  L000003              2   2023-09-01           51808.14        51859.18   
2  L000003              3   2023-10-01           51808.14        51457.49   
3  L000003              4   2023-11-01           51808.14        50825.00   
4  L000003              5   2023-12-01           51808.14        52244.67   

   outstanding_balance  interest_component  principal_component  dpd  \
0           4290894.80            15742.61             36969.32    0   
1           4254643.75            15608.13             36251.05    0   
2           4218662.53            15476.27             35981.22    0 

PD and LGD Model Development by fitting ML techniques into the dataframes

In [20]:
# 1. Load data
loans = pd.read_csv('loan_portfolio.csv')
port_metrics = pd.read_csv('portfolio_metrics.csv')
macro_scenarios = pd.read_csv('macro_stress_scenarios.csv')

print("Loaded datasets successfully.")

# Feature engineering for PD model
# Features: credit_score, leverage, interest_coverage, debt_to_equity, coupon_rate, maturity_months, sector, loan_type, collateral
loans_encoded = pd.get_dummies(loans, columns=['sector', 'loan_type', 'collateral'], drop_first=True)

feature_cols_pd = ['credit_score', 'leverage', 'interest_coverage', 'debt_to_equity', 'coupon_rate', 'maturity_months'] + \
                  [c for c in loans_encoded.columns if c.startswith(('sector_', 'loan_type_', 'collateral_'))]

X_pd = loans_encoded[feature_cols_pd]
y_pd = loans_encoded['defaulted']

X_train_pd, X_test_pd, y_train_pd, y_test_pd = train_test_split(X_pd, y_pd, test_size=0.2, random_state=42, stratify=y_pd)

# Candidate Models for PD:
# 1. Logistic Regression (Benchmark / Regulatory preferred)
# 2. Random Forest Classifier
# 3. Gradient Boosting Classifier

#Fitting a Logistic Regression Model
lr_pd = LogisticRegression(max_iter=1000, C=1.0)
lr_pd.fit(X_train_pd, y_train_pd)
lr_pd_preds = lr_pd.predict_proba(X_test_pd)[:, 1]

#Fitting a RandomForestClassifier Model
rf_pd = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=8)
rf_pd.fit(X_train_pd, y_train_pd)
rf_pd_preds = rf_pd.predict_proba(X_test_pd)[:, 1]

#Fitting a Gradient Boosting Classifier Model
gb_pd = GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=4)
gb_pd.fit(X_train_pd, y_train_pd)
gb_pd_preds = gb_pd.predict_proba(X_test_pd)[:, 1]

#Evaluating the PD Model using AUC, Gini, and Brier
print("\n=== PD MODEL EVALUATION ===")
print(f"Logistic Regression - AUC: {roc_auc_score(y_test_pd, lr_pd_preds):.4f}, Gini: {2*roc_auc_score(y_test_pd, lr_pd_preds)-1:.4f}, Brier: {brier_score_loss(y_test_pd, lr_pd_preds):.5f}")
print(f"Random Forest       - AUC: {roc_auc_score(y_test_pd, rf_pd_preds):.4f}, Gini: {2*roc_auc_score(y_test_pd, rf_pd_preds)-1:.4f}, Brier: {brier_score_loss(y_test_pd, rf_pd_preds):.5f}")
print(f"Gradient Boosting   - AUC: {roc_auc_score(y_test_pd, gb_pd_preds):.4f}, Gini: {2*roc_auc_score(y_test_pd, gb_pd_preds)-1:.4f}, Brier: {brier_score_loss(y_test_pd, gb_pd_preds):.5f}")

# LGD Modeling (on defaulted loans)
df_def = loans_encoded[loans_encoded['defaulted'] == 1].copy()
y_lgd = df_def['lgd'] # or 1 - recovery_rate
X_lgd = df_def[feature_cols_pd]

X_train_lgd, X_test_lgd, y_train_lgd, y_test_lgd = train_test_split(X_lgd, y_lgd, test_size=0.2, random_state=42)

# Candidate Models for LGD:
# 1. Linear Regression (OLS)
# 2. Ridge Regression
# 3. Gradient Boosting Regressor

#Fitting a linear regression model for LGD
ols_lgd = LinearRegression()
ols_lgd.fit(X_train_lgd, y_train_lgd)
ols_lgd_preds = np.clip(ols_lgd.predict(X_test_lgd), 0, 1)

#Fitting a ridge model for LGD
ridge_lgd = Ridge(alpha=1.0)
ridge_lgd.fit(X_train_lgd, y_train_lgd)
ridge_lgd_preds = np.clip(ridge_lgd.predict(X_test_lgd), 0, 1)

#Fitting a Gradient Booster Model for LGD
gb_lgd = GradientBoostingRegressor(n_estimators=100, random_state=42, max_depth=3)
gb_lgd.fit(X_train_lgd, y_train_lgd)
gb_lgd_preds = np.clip(gb_lgd.predict(X_test_lgd), 0, 1)

#Evaluating the LGD models using RMSE, Mean Absoluter error, and R-squared
print("\n=== LGD MODEL EVALUATION ===")
print(f"OLS Regression - RMSE: {np.sqrt(mean_squared_error(y_test_lgd, ols_lgd_preds)):.4f}, MAE: {mean_absolute_error(y_test_lgd, ols_lgd_preds):.4f}, R2: {r2_score(y_test_lgd, ols_lgd_preds):.4f}")
print(f"Ridge Regr     - RMSE: {np.sqrt(mean_squared_error(y_test_lgd, ridge_lgd_preds)):.4f}, MAE: {mean_absolute_error(y_test_lgd, ridge_lgd_preds):.4f}, R2: {r2_score(y_test_lgd, ridge_lgd_preds):.4f}")
print(f"Gradient Boost - RMSE: {np.sqrt(mean_squared_error(y_test_lgd, gb_lgd_preds)):.4f}, MAE: {mean_absolute_error(y_test_lgd, gb_lgd_preds):.4f}, R2: {r2_score(y_test_lgd, gb_lgd_preds):.4f}")

Loaded datasets successfully.


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



=== PD MODEL EVALUATION ===
Logistic Regression - AUC: 0.7148, Gini: 0.4296, Brier: 0.10982
Random Forest       - AUC: 0.7261, Gini: 0.4521, Brier: 0.10950
Gradient Boosting   - AUC: 0.7375, Gini: 0.4750, Brier: 0.10791

=== LGD MODEL EVALUATION ===
OLS Regression - RMSE: 0.0925, MAE: 0.0754, R2: 0.5937
Ridge Regr     - RMSE: 0.0925, MAE: 0.0754, R2: 0.5937
Gradient Boost - RMSE: 0.0943, MAE: 0.0768, R2: 0.5781


In [21]:
# Let's scale data for Logistic Regression / OLS and re-evaluate accurately

scaler = StandardScaler()
X_train_pd_scaled = scaler.fit_transform(X_train_pd)
X_test_pd_scaled = scaler.transform(X_test_pd)

lr_pd = LogisticRegression(max_iter=2000, C=1.0)
lr_pd.fit(X_train_pd_scaled, y_train_pd)
lr_pd_preds = lr_pd.predict_proba(X_test_pd_scaled)[:, 1]

rf_pd = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=8)
rf_pd.fit(X_train_pd, y_train_pd)
rf_pd_preds = rf_pd.predict_proba(X_test_pd)[:, 1]

gb_pd = GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=4)
gb_pd.fit(X_train_pd, y_train_pd)
gb_pd_preds = gb_pd.predict_proba(X_test_pd)[:, 1]

print("\n=== RE-EVALUATED PD MODEL METRICS ===")
print(f"Logistic Regression - AUC: {roc_auc_score(y_test_pd, lr_pd_preds):.4f}, Gini: {2*roc_auc_score(y_test_pd, lr_pd_preds)-1:.4f}, Brier: {brier_score_loss(y_test_pd, lr_pd_preds):.5f}")
print(f"Random Forest       - AUC: {roc_auc_score(y_test_pd, rf_pd_preds):.4f}, Gini: {2*roc_auc_score(y_test_pd, rf_pd_preds)-1:.4f}, Brier: {brier_score_loss(y_test_pd, rf_pd_preds):.5f}")
print(f"Gradient Boosting   - AUC: {roc_auc_score(y_test_pd, gb_pd_preds):.4f}, Gini: {2*roc_auc_score(y_test_pd, gb_pd_preds)-1:.4f}, Brier: {brier_score_loss(y_test_pd, gb_pd_preds):.5f}")

# LGD models with scaling for OLS / Ridge
scaler_lgd = StandardScaler()
X_train_lgd_scaled = scaler_lgd.fit_transform(X_train_lgd)
X_test_lgd_scaled = scaler_lgd.transform(X_test_lgd)

ols_lgd = LinearRegression()
ols_lgd.fit(X_train_lgd_scaled, y_train_lgd)
ols_lgd_preds = np.clip(ols_lgd.predict(X_test_lgd_scaled), 0, 1)

ridge_lgd = Ridge(alpha=10.0)
ridge_lgd.fit(X_train_lgd_scaled, y_train_lgd)
ridge_lgd_preds = np.clip(ridge_lgd.predict(X_test_lgd_scaled), 0, 1)

gb_lgd = GradientBoostingRegressor(n_estimators=100, random_state=42, max_depth=3)
gb_lgd.fit(X_train_lgd, y_train_lgd)
gb_lgd_preds = np.clip(gb_lgd.predict(X_test_lgd), 0, 1)

print("\n=== RE-EVALUATED LGD MODEL METRICS ===")
print(f"OLS Regression - RMSE: {np.sqrt(mean_squared_error(y_test_lgd, ols_lgd_preds)):.4f}, MAE: {mean_absolute_error(y_test_lgd, ols_lgd_preds):.4f}, R2: {r2_score(y_test_lgd, ols_lgd_preds):.4f}")
print(f"Ridge Regr     - RMSE: {np.sqrt(mean_squared_error(y_test_lgd, ridge_lgd_preds)):.4f}, MAE: {mean_absolute_error(y_test_lgd, ridge_lgd_preds):.4f}, R2: {r2_score(y_test_lgd, ridge_lgd_preds):.4f}")
print(f"Gradient Boost - RMSE: {np.sqrt(mean_squared_error(y_test_lgd, gb_lgd_preds)):.4f}, MAE: {mean_absolute_error(y_test_lgd, gb_lgd_preds):.4f}, R2: {r2_score(y_test_lgd, gb_lgd_preds):.4f}")


=== RE-EVALUATED PD MODEL METRICS ===
Logistic Regression - AUC: 0.7159, Gini: 0.4319, Brier: 0.10971
Random Forest       - AUC: 0.7261, Gini: 0.4521, Brier: 0.10950
Gradient Boosting   - AUC: 0.7375, Gini: 0.4750, Brier: 0.10791

=== RE-EVALUATED LGD MODEL METRICS ===
OLS Regression - RMSE: 0.0925, MAE: 0.0754, R2: 0.5937
Ridge Regr     - RMSE: 0.0925, MAE: 0.0754, R2: 0.5937
Gradient Boost - RMSE: 0.0943, MAE: 0.0768, R2: 0.5781


Calculation of EAD and EL

In [22]:
#Loan Portfolio Summary for EAD Calculation

print("Loan portfolio EAD summary:")
print(loans[['ead', 'pd_annual', 'lgd', 'el', 'loss_given_default']].head(10))

Loan portfolio EAD summary:
          ead  pd_annual     lgd         el  loss_given_default
0  1035611.99   0.041076  0.7858   33426.80                 NaN
1   800786.92   0.000887  0.7034     499.82                 NaN
2  4327864.12   0.002622  0.5503    6243.54                 NaN
3  3810461.19   0.041679  0.3541   56237.33                 NaN
4  2925538.53   0.136071  0.6576  261778.69                 NaN
5    76645.49   0.041237  0.5772    1824.29                 NaN
6   732703.69   0.000427  0.6496     203.07                 NaN
7    50000.00   0.002544  0.4838      61.53            24190.00
8   442881.00   0.000890  0.4484     176.74                 NaN
9    60656.97   0.002568  0.5130      79.91            31117.03


In [23]:
# Let's check how EL is computed in loan_portfolio.csv
calculated_el = loans['ead'] * loans['pd_annual'] * loans['lgd']
diff = np.abs(loans['el'] - calculated_el).max()
print("Max difference between stored el and ead * pd * lgd:", diff)

Max difference between stored el and ead * pd * lgd: 68.96280642715283


In [24]:
# Let's check macro_stress_scenarios.csv for EAD and EL under stress
macro = pd.read_csv('macro_stress_scenarios.csv')
print("\nMacro stress scenarios columns:", macro.columns.tolist())
print(macro[['scenario', 'sector', 'base_pd', 'stressed_pd', 'base_lgd', 'stressed_lgd', 'total_ead', 'expected_loss_base', 'expected_loss_stress']].head())


Macro stress scenarios columns: ['scenario', 'gdp_shock_pp', 'unemp_shock_pp', 'rate_shock_pp', 'credit_spread_bps', 'sector', 'base_pd', 'stressed_pd', 'pd_uplift_pp', 'pd_multiplier', 'base_lgd', 'stressed_lgd', 'total_ead', 'expected_loss_base', 'expected_loss_stress', 'el_increase_pct']
   scenario       sector   base_pd  stressed_pd  base_lgd  stressed_lgd  \
0  baseline   Financials  0.022308     0.022308      0.45          0.45   
1  baseline  Real_Estate  0.022656     0.022656      0.40          0.40   
2  baseline     Consumer  0.021524     0.021524      0.55          0.55   
3  baseline  Industrials  0.021915     0.021915      0.50          0.50   
4  baseline   Technology  0.023575     0.023575      0.60          0.60   

      total_ead  expected_loss_base  expected_loss_stress  
0  3.016669e+10        3.161070e+08          3.028334e+08  
1  2.530543e+10        2.617922e+08          2.293309e+08  
2  5.049910e+09        6.904421e+07          5.978082e+07  
3  1.571215e+10 

In [25]:
print(loans['ead'].describe())
print(loans.columns.tolist())

count    5.000000e+04
mean     3.298605e+06
std      6.661943e+06
min      5.000000e+04
25%      5.482685e+05
50%      1.389712e+06
75%      3.425439e+06
max      3.611778e+08
Name: ead, dtype: float64
['loan_id', 'origination_date', 'maturity_date', 'maturity_months', 'sector', 'loan_type', 'collateral', 'initial_rating', 'credit_score', 'ead', 'coupon_rate', 'leverage', 'interest_coverage', 'debt_to_equity', 'pd_annual', 'lgd', 'el', 'unexpected_loss', 'rwa', 'defaulted', 'default_date', 'survival_months', 'recovery_rate', 'loss_given_default']


In [26]:


#Feature Engineering & Preparation
loans_encoded = pd.get_dummies(loans, columns=['sector', 'loan_type', 'collateral'], drop_first=True)
feature_cols = ['credit_score', 'leverage', 'interest_coverage', 'debt_to_equity', 'coupon_rate', 'maturity_months'] + \
               [c for c in loans_encoded.columns if c.startswith(('sector_', 'loan_type_', 'collateral_'))]

X = loans_encoded[feature_cols]
y_pd = loans_encoded['defaulted']

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit Champion PD Model (Logistic Regression / GBM)
lr_pd = LogisticRegression(max_iter=2000, C=1.0)
lr_pd.fit(X_scaled, y_pd)
loans['predicted_pd'] = lr_pd.predict_proba(X_scaled)[:, 1]

# Fit Champion LGD Model (OLS Regression on defaulted loans)
df_def = loans_encoded[loans_encoded['defaulted'] == 1].copy()
X_def = df_def[feature_cols]
y_lgd = df_def['lgd']

scaler_lgd = StandardScaler()
X_def_scaled = scaler_lgd.fit_transform(X_def)

ols_lgd = LinearRegression()
ols_lgd.fit(X_def_scaled, y_lgd)

# Predict LGD for all loans
X_all_scaled = scaler_lgd.transform(X)
loans['predicted_lgd'] = np.clip(ols_lgd.predict(X_all_scaled), 0, 1)

# EAD Integration (Using actual EAD from portfolio / CCF adjustment for revolvers)
# For term loans / mortgages, EAD equals current outstanding balance ('ead')
loans['modeled_ead'] = loans['ead']

# Calculate Expected Loss (EL = PD * LGD * EAD)
loans['calculated_el'] = loans['predicted_pd'] * loans['predicted_lgd'] * loans['modeled_ead']

# Summary Statistics
total_portfolio_ead = loans['modeled_ead'].sum()
total_portfolio_el = loans['calculated_el'].sum()
portfolio_el_rate = total_portfolio_el / total_portfolio_ead

print(f"=== PORTFOLIO EXPECTED LOSS SUMMARY ===")
print(f"Total Portfolio EAD : ${total_portfolio_ead:,.2f}")
print(f"Total Expected Loss  : ${total_portfolio_el:,.2f}")
print(f"Portfolio EL Rate    : {portfolio_el_rate * 100:.4f}%")
print(loans[['loan_id', 'predicted_pd', 'predicted_lgd', 'modeled_ead', 'calculated_el']].head(5))

=== PORTFOLIO EXPECTED LOSS SUMMARY ===
Total Portfolio EAD : $164,930,233,732.19
Total Expected Loss  : $12,019,647,126.94
Portfolio EL Rate    : 7.2877%
   loan_id  predicted_pd  predicted_lgd  modeled_ead  calculated_el
0  L000001      0.101586       0.557629   1035611.99   58664.822375
1  L000002      0.130248       0.646806    800786.92   67462.584728
2  L000003      0.120759       0.359463   4327864.12  187865.015966
3  L000004      0.483544       0.351665   3810461.19  647952.818985
4  L000005      0.429971       0.622026   2925538.53  782445.456160
